In [1]:
import pandas as pd, re, torch
from google.colab import drive
from transformers import pipeline

drive.mount('/content/drive')
D = "/content/drive/MyDrive/My_Projects/EmotionLens/"
df = pd.read_csv(D + "comments_export.csv")
df["text"] = df["text"].fillna("").astype(str)

clf = pipeline("sentiment-analysis",
               model="cardiffnlp/twitter-roberta-base-sentiment-latest",
               device=0 if torch.cuda.is_available() else -1,
               batch_size=64, truncation=True, max_length=128, top_k=None)
out = clf(df["text"].tolist())

p = pd.DataFrame([{s["label"].lower(): s["score"] for s in scores} for scores in out])
df["p_negative"], df["p_neutral"], df["p_positive"] = p["negative"], p["neutral"], p["positive"]
df["label"] = p.idxmax(axis=1)
df["confidence"] = p.max(axis=1)
second = p.apply(lambda r: sorted(r.values)[-2], axis=1)
df["margin"] = df["confidence"] - second   # small margin = model is torn

Mounted at /content/drive


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [2]:
rev = re.compile(r"#(not|sarcasm|sarcastic|irony|ironic)\b", re.I)
df["reverse_tag"] = df["text"].apply(lambda t: bool(rev.search(t)))

hindi_words = {"hai", "hain", "nahi", "nhi", "ki", "ko", "ka", "ke", "bhi", "toh", "kya",
               "ye", "yeh", "se", "ek", "aur", "pe", "mein", "tha", "thi", "kar", "koi",
               "sab", "bhai", "jyada", "bahut"}

def lang(t):
    if sum('\u0900' <= ch <= '\u097f' for ch in t) >= 3:
        return "devanagari"
    hits = sum(w in hindi_words for w in re.findall(r"[a-z]+", t.lower()))
    return "hinglish" if hits >= 2 else "english/other"

df["lang_guess"] = df["text"].apply(lang)
df.to_csv(D + "comments_scored.csv", index=False, encoding="utf-8-sig")

print(df.groupby(["theme", "label"]).size().unstack(fill_value=0))
print()
print(df["lang_guess"].value_counts())
print()
print(df.groupby("lang_guess")[["confidence", "margin"]].mean().round(3))
print()
print("comments with #not / #sarcasm / #irony:", int(df["reverse_tag"].sum()))

label  negative  neutral  positive
theme                             
A            42      109        69
B           120       76        23

lang_guess
english/other    415
hinglish          19
devanagari         5
Name: count, dtype: int64

               confidence  margin
lang_guess                       
devanagari          0.794   0.639
english/other       0.742   0.523
hinglish            0.797   0.637

comments with #not / #sarcasm / #irony: 0
